# NN_09 — findings

`BUILD_SPEC.md` §7, row 09: the consolidated write-up, including a null if that is the result, ready to transcribe into `research/strategies/multi_factor_ranking_ml/docs/reproducibility_findings.md`.

**This notebook trains nothing, computes no new metric, and makes no new claim.** It loads the run artifacts that the executed `NN_05` and `NN_06` saved, verifies their provenance against the real-data slice and against the executed notebooks, reproduces the summary numbers those notebooks already printed (as a consistency check between artifact and record, not as a new measurement), and prints a draft findings entry. Nothing under `src/atlas_quant/` was changed anywhere in this series, and nothing is written to the findings document by this notebook — transcription is a separate, human step.

**Stop condition.** `NN_06`, the decisive FiLM ablation, was **null on this slice**. Per `BUILD_SPEC.md` §9 the series stopped there: `NN_07` (seed variance) and `NN_08` (baseline comparison and ensemble) were **not built**, because both are performance-seeking follow-ups whose premise was not established. `NN_09` therefore follows `NN_06` directly; it introduces no result of its own.

**Data mode: REAL DATA.** The artifacts were produced from the cached real-data slice (`rd.slice_bundle()`) and the real daily closes; their slice fingerprint and config identity are checked below before any number is shown. If the raw exports are absent, every computing cell prints the explicit unavailability notice and does nothing.

Kernel: `atlasquant-venv` (= `.venv/bin/python`).

In [1]:
import nn_common as nc

rd = nc.rd
AVAILABLE = nc.provenance_banner("NN_09_findings.ipynb")

neural_ranker / NN_09_findings.ipynb
DATA MODE: REAL DATA.

Source: Bloomberg CSV exports under this repository's gitignored raw-data
root for multi_factor_ranking_ml -- 1,520 tickers, quarterly fundamentals
from 1980 onward, plus daily closes and macro series.

Every number below is computed from that real data through production
AtlasQuant code. No synthetic, simulated, imputed, or approximated values
are used anywhere in this notebook series.

Bounded slice: quarterly evaluation cycles 2020-10-01 .. 2024-01-01
(a genuine subset of the history, chosen so each notebook runs quickly --
not a downsample or an approximation of a longer run).

SOURCE MANIFEST (SHA-256 of every raw CSV the pipeline reads)
  raw root: /Users/gavinhussey/Downloads/AtlasQuant/data/raw/multi_factor_ranking_ml
  fundamentals_quarterly.csv      281.3 MB  2026-08-31 23:01:10  3315e7104e640fc790b6e12c66b0f6a122cf66bb97a8ed3124abaced0c86b48a
  Close_Price.csv                  67.6 MB  2026-08-31 23:07:42  2723e56e4

libraries         : python=3.14.5, torch=2.13.0, numpy=2.5.2, pandas=3.0.5, scipy=1.18.0, lightgbm=4.7.0


## The record: which notebooks exist, executed clean, and which were deliberately not built

In [2]:
import json

import nbformat
import numpy as np

SERIES = {
    "NN_00_setup_and_provenance": "environment, real-data SHA-256 manifest, seed policy, device",
    "NN_01_tensor_construction": "one cycle -> X, mask, sector, quarter, m (6,), y; mask == missing_features exactly",
    "NN_02_normalization": "five raw-level features rank-normalized (production function, per cycle); everything else passed through",
    "NN_03_pit_verification": "later cycles removed -> byte-identical tensors on all 13 cycles; negative control fails as it should",
    "NN_04_baseline_model": "encoder + embeddings + FiLM + ListNet; capacity check; literal spec target measured degenerate",
    "NN_05_walkforward": "walk-forward on the baseline's cycles, graded target at temperature 0.5, CPU, 5 seeds",
    "NN_06_film_ablation": "FiLM vs FiLM-removed, paired per seed and cycle: NULL on this slice",
}
NOT_BUILT = {
    "NN_07_seed_variance": "not built: stop condition after the null in NN_06 (seed variance is already reported there)",
    "NN_08_baseline_comparison": "not built: stop condition after the null in NN_06",
}
print(f"{'notebook':<32}{'exists':>7}{'code cells':>11}{'executed in order':>19}{'errors':>8}  established")
for name, what in SERIES.items():
    path = nc.NEURAL_RANKER_DIR / f"{name}.ipynb"
    nb = nbformat.read(path, as_version=4)
    nbformat.validate(nb)
    counts = [c.get("execution_count") for c in nb.cells if c.cell_type == "code"]
    errors = sum(1 for c in nb.cells if c.cell_type == "code" for o in c.get("outputs", []) if o["output_type"] == "error")
    in_order = counts == list(range(1, len(counts) + 1))
    print(f"{name:<32}{'yes':>7}{len(counts):>11}{str(in_order):>19}{errors:>8}  {what}")
    assert in_order and errors == 0, f"{name} is not a clean top-to-bottom execution"
for name, why in NOT_BUILT.items():
    print(f"{name:<32}{'no':>7}{'-':>11}{'-':>19}{'-':>8}  {why}")
    assert not (nc.NEURAL_RANKER_DIR / f"{name}.ipynb").exists()

notebook                         exists code cells  executed in order  errors  established
NN_00_setup_and_provenance          yes         10               True       0  environment, real-data SHA-256 manifest, seed policy, device
NN_01_tensor_construction           yes         11               True       0  one cycle -> X, mask, sector, quarter, m (6,), y; mask == missing_features exactly
NN_02_normalization                 yes         13               True       0  five raw-level features rank-normalized (production function, per cycle); everything else passed through
NN_03_pit_verification              yes          7               True       0  later cycles removed -> byte-identical tensors on all 13 cycles; negative control fails as it should
NN_04_baseline_model                yes         10               True       0  encoder + embeddings + FiLM + ListNet; capacity check; literal spec target measured degenerate
NN_05_walkforward                   yes         11               True

## The artifacts, verified before use

Both artifacts must carry this slice's fingerprint and config identity, the same fixed protocol (temperature 0.5, CPU, seeds 0–4), and the baseline ICs stored in them must equal the slice's own `ic_result` bit for bit.

In [3]:
if rd.require_data():
    bundle = rd.slice_bundle()
    cfg = rd.config()
    OUT_DIR = nc.NEURAL_RANKER_DIR / "outputs"
    NN05 = json.loads((OUT_DIR / "NN_05_walkforward.json").read_text())
    NN06 = json.loads((OUT_DIR / "NN_06_film_ablation.json").read_text())
    baseline = {r.cycle.quarter_start.isoformat(): r for r in bundle.ic_result.cycle_results}
    for label, art in (("NN_05", NN05), ("NN_06", NN06)):
        assert art["slice_fingerprint"] == bundle.fingerprint and art["config_identity"] == cfg.identity()
        assert art["device"] == "cpu" and tuple(art["seeds"]) == nc.SEEDS and art["protocol"]["temperature"] == 0.5
        print(f"{label}: notebook {art['notebook']}, git {art['git_commit'][:12]}, fingerprint {art['slice_fingerprint']}, "
              f"device {art['device']}, seeds {art['seeds']}, temperature {art['protocol']['temperature']}, records {len(art['results'])}")
    assert NN05["protocol"] == NN06["protocol"]
    for q, ic in NN05["baseline_ic"].items():
        assert ic == baseline[q].ic
    MEASURED = sorted(NN05["baseline_ic"])
    print(f"\nprotocol (shared): {NN05['protocol']}")
    print(f"measured cycles: {len(MEASURED)} ({MEASURED[0]} .. {MEASURED[-1]}); baseline ICs in the artifact equal the slice's stored ICs bit for bit: True")
    print(f"NN_06 stored verdict: {NN06['verdict']}")

NN_05: notebook NN_05_walkforward.ipynb, git 7b7f2681146b, fingerprint 168fedca9e7b334fdf7fb0e4b8a15cb2, device cpu, seeds [0, 1, 2, 3, 4], temperature 0.5, records 45
NN_06: notebook NN_06_film_ablation.ipynb, git 7b7f2681146b, fingerprint 168fedca9e7b334fdf7fb0e4b8a15cb2, device cpu, seeds [0, 1, 2, 3, 4], temperature 0.5, records 90

protocol (shared): {'lr': 0.001, 'weight_decay': 0.01, 'max_steps': 1000, 'eval_every': 10, 'patience': 20, 'temperature': 0.5, 'target': 'softmax(((2**y - 1) / (2**9 - 1)) / 0.5)', 'holdout': 'most recent included training quarter'}
measured cycles: 8 (2022-01-01 .. 2023-10-01); baseline ICs in the artifact equal the slice's stored ICs bit for bit: True
NN_06 stored verdict: NULL on this slice: the FiLM effect is not distinguishable from zero at 8 cycles


## `NN_05` — the walk-forward, reproduced from its artifact

The per-fit ICs are read from the artifact and summarized with the same arithmetic `NN_05` used. The reference values are the numbers `NN_05` printed; the assertion ties the artifact to the record.

In [4]:
if rd.require_data():
    rec05 = {(r["target"], r["seed"]): r for r in NN05["results"]}
    seed_mean05 = {q: float(np.mean([rec05[(q, s)]["ic"] for s in nc.SEEDS])) for q in MEASURED}
    base_ic = {q: NN05["baseline_ic"][q] for q in MEASURED}
    summary = lambda v: (float(np.mean(v)), float(np.std(v)), float(np.mean(np.array(v) > 0)))
    b = summary([base_ic[q] for q in MEASURED])
    n = summary([seed_mean05[q] for q in MEASURED])
    paired05 = np.array([seed_mean05[q] - base_ic[q] for q in MEASURED])
    per_seed05 = {s: float(np.mean([rec05[(q, s)]["ic"] for q in MEASURED])) for s in nc.SEEDS}
    best_steps05 = [r["best_step"] for r in NN05["results"]]

    print(f"{'cycle':<12}{'LightGBM':>10}{'NN seed-mean':>14}{'NN - LGBM':>11}")
    for q in MEASURED:
        print(f"{q:<12}{base_ic[q]:>+10.4f}{seed_mean05[q]:>+14.4f}{seed_mean05[q] - base_ic[q]:>+11.4f}")
    print(f"\nLightGBM baseline : mean IC {b[0]:+.4f}, std {b[1]:.4f}, hit rate {b[2]:.3f}")
    print(f"NN seed-mean IC   : mean IC {n[0]:+.4f}, std {n[1]:.4f}, hit rate {n[2]:.3f}")
    print(f"per-seed mean IC  : " + ", ".join(f"seed {s} {v:+.4f}" for s, v in per_seed05.items()))
    print(f"paired NN - LGBM  : mean {paired05.mean():+.4f}, std {paired05.std():.4f}, NN higher in {int((paired05 > 0).sum())} of {len(paired05)} cycles")
    print(f"early stopping    : best step min {min(best_steps05)}, median {int(np.median(best_steps05))}, max {max(best_steps05)} over {len(best_steps05)} fits")

    # the numbers NN_05 printed, as its record
    assert (round(b[0], 4), round(b[1], 4), round(b[2], 3)) == (0.0275, 0.0916, 0.750)
    assert (round(n[0], 4), round(n[1], 4), round(n[2], 3)) == (0.0492, 0.0751, 0.875)
    assert (round(float(paired05.mean()), 4), round(float(paired05.std()), 4), int((paired05 > 0).sum())) == (0.0218, 0.0506, 6)
    assert (min(best_steps05), max(best_steps05)) == (10, 50)
    print("\nartifact reproduces the numbers NN_05 printed: True")

cycle         LightGBM  NN seed-mean  NN - LGBM
2022-01-01     -0.0261       +0.0759    +0.1020
2022-04-01     -0.1900       -0.1422    +0.0478
2022-07-01     +0.0618       +0.1046    +0.0428
2022-10-01     +0.0583       +0.0633    +0.0050
2023-01-01     +0.0708       +0.0584    -0.0124
2023-04-01     +0.0335       +0.0874    +0.0539
2023-07-01     +0.0877       +0.1029    +0.0153
2023-10-01     +0.1240       +0.0436    -0.0803

LightGBM baseline : mean IC +0.0275, std 0.0916, hit rate 0.750
NN seed-mean IC   : mean IC +0.0492, std 0.0751, hit rate 0.875
per-seed mean IC  : seed 0 +0.0498, seed 1 +0.0331, seed 2 +0.0522, seed 3 +0.0637, seed 4 +0.0474
paired NN - LGBM  : mean +0.0218, std 0.0506, NN higher in 6 of 8 cycles
early stopping    : best step min 10, median 20, max 50 over 45 fits

artifact reproduces the numbers NN_05 printed: True


## `NN_06` — the ablation, reproduced from its artifact

Per-cycle seed-mean ICs of both arms and their paired difference are read from the artifact; the paired *t*-test is recomputed with the same call `NN_06` used and must equal the stored test summary and the numbers `NN_06` printed.

In [5]:
if rd.require_data():
    from scipy import stats

    rec06 = {(r["arm"], r["target"], r["seed"]): r for r in NN06["results"]}
    film = {q: float(np.mean([rec06[("film", q, s)]["ic"] for s in nc.SEEDS])) for q in MEASURED}
    nofilm = {q: float(np.mean([rec06[("no_film", q, s)]["ic"] for s in nc.SEEDS])) for q in MEASURED}
    D = np.array([film[q] - nofilm[q] for q in MEASURED])
    t_stat, p_value = stats.ttest_rel([film[q] for q in MEASURED], [nofilm[q] for q in MEASURED])
    per_seed_diff = {s: float(np.mean([rec06[("film", q, s)]["ic"] - rec06[("no_film", q, s)]["ic"] for q in MEASURED])) for s in nc.SEEDS}
    seed_noise = float(np.std(list(per_seed_diff.values()), ddof=1))

    print(f"{'cycle':<12}{'LightGBM':>10}{'FiLM':>9}{'no-FiLM':>9}{'FiLM - noFiLM':>15}")
    for q in MEASURED:
        print(f"{q:<12}{base_ic[q]:>+10.4f}{film[q]:>+9.4f}{nofilm[q]:>+9.4f}{D[MEASURED.index(q)]:>+15.4f}")
    print(f"\npaired t-test across {len(D)} cycles: mean difference {D.mean():+.4f}, std {D.std(ddof=1):.4f}, SE {D.std(ddof=1) / np.sqrt(len(D)):.4f}, "
          f"t = {t_stat:+.3f}, p = {p_value:.3f}, FiLM higher in {int((D > 0).sum())} of {len(D)} cycles")
    print("per-seed mean difference: " + ", ".join(f"seed {s} {v:+.4f}" for s, v in per_seed_diff.items()) + f"; std across seeds {seed_noise:.4f}")
    print(f"stored test summary     : {NN06['paired_test']}")
    print(f"stored verdict          : {NN06['verdict']}")

    stored = NN06["paired_test"]
    assert abs(stored["mean_diff"] - D.mean()) < 1e-12 and abs(stored["t"] - t_stat) < 1e-9 and abs(stored["p"] - p_value) < 1e-9 and stored["n_cycles"] == len(D)
    assert (round(float(D.mean()), 4), round(float(t_stat), 3), round(float(p_value), 3), int((D > 0).sum())) == (-0.0094, -1.271, 0.244, 4)
    assert NN06["verdict"].startswith("NULL")
    VERDICT = NN06["verdict"]
    print("\nartifact reproduces the numbers NN_06 printed, and its stored test summary: True")

cycle         LightGBM     FiLM  no-FiLM  FiLM - noFiLM
2022-01-01     -0.0261  +0.0759  +0.1179        -0.0420
2022-04-01     -0.1900  -0.1422  -0.1002        -0.0420
2022-07-01     +0.0618  +0.1046  +0.1044        +0.0002
2022-10-01     +0.0583  +0.0633  +0.0641        -0.0008
2023-01-01     +0.0708  +0.0584  +0.0473        +0.0111
2023-04-01     +0.0335  +0.0874  +0.0974        -0.0099
2023-07-01     +0.0877  +0.1029  +0.0985        +0.0044
2023-10-01     +0.1240  +0.0436  +0.0399        +0.0037

paired t-test across 8 cycles: mean difference -0.0094, std 0.0210, SE 0.0074, t = -1.271, p = 0.244, FiLM higher in 4 of 8 cycles
per-seed mean difference: seed 0 -0.0062, seed 1 -0.0274, seed 2 -0.0238, seed 3 -0.0045, seed 4 +0.0149; std across seeds 0.0170
stored test summary     : {'mean_diff': -0.009416735796216805, 'std_diff': 0.020956520486257735, 't': -1.270943378699421, 'p': 0.2443525090135195, 'n_cycles': 8}
stored verdict          : NULL on this slice: the FiLM effect is not dis

## Draft entry for `reproducibility_findings.md`

Printed, not written: transcription is a separate human step. None of the five provenance classifications (`source_specification_required`, `data_provenance_required`, `implementation_bug`, `data_quality_issue`, `expected_legacy_difference`) describes a research null, and no strategy logic changed, so the heading carries no classification tag rather than a misapplied one.

In [6]:
if rd.require_data():
    entry = f'''
## 2026-09-09 — neural ranker experiment (`neural_ranker/`): FiLM macro conditioning is null on the cached slice

**Question.** Can a macro-conditioned neural ranker (FiLM: the quarter's 6-value macro vector modulating a per-stock
encoder, `neural_ranker/OUTLINE.md` §1) express something the LambdaRank LightGBM baseline structurally cannot, and does
it show up as out-of-sample IC? Built and measured as notebooks `NN_00`–`NN_06` (+ this consolidation, `NN_09`) on the
cached real-data slice 2020-10-01 .. 2024-01-01 (14 cycles, 9 trained, 8 with a measured IC; slice fingerprint
{bundle.fingerprint}; git {NN06['git_commit'][:12]}). Nothing under `src/atlas_quant/` was modified.

**Answer: null on this slice.** Paired per seed and per cycle (same seeds, same windows, same Materials-excluded names,
same real forward returns), FiLM minus FiLM-removed IC averages {D.mean():+.4f} over the 8 measured cycles
(std {D.std(ddof=1):.4f}, t = {t_stat:+.3f}, p = {p_value:.3f}, FiLM higher in {int((D > 0).sum())} of 8 cycles; per-seed mean difference
{min(per_seed_diff.values()):+.4f} .. {max(per_seed_diff.values()):+.4f}, std across seeds {seed_noise:.4f}). By the verdict rule registered before the run
(helps/hurts at p < 0.05, else null), the result is null. No production value is claimed for either arm.

**Walk-forward context (`NN_05`, not a claim).** Under the same fixed protocol the FiLM model's seed-mean IC over the
8 cycles is {n[0]:+.4f} (std {n[1]:.4f}, hit rate {n[2]:.3f}) against the stored LightGBM baseline's {b[0]:+.4f} (std {b[1]:.4f},
hit rate {b[2]:.3f}); paired NN minus LightGBM {paired05.mean():+.4f} (std {paired05.std():.4f}), NN higher in {int((paired05 > 0).sum())} of 8 cycles; per-seed
mean IC {min(per_seed05.values()):+.4f} .. {max(per_seed05.values()):+.4f}. Eight cycles is far thinner than the 22-cycle full-history baseline and no
significance test was run on this comparison.

**Fixed protocol decisions, recorded as decisions, not tuned results.** (1) Rank-normalize only the five raw-level
features the production pipeline already ranks (`market_cap`, `volume`, `net_debt`, `adjusted_net_debt`,
`analyst_target_price`), honoring `BUILD_SPEC.md` §5 rule 2 over §4's code block; `normalize_cycle` is a measured
no-op on the cached slice. (2) Fixed, non-fitted encoder input: identity on those five, `sign(x)·log1p(|x|)` on the
other 58 stock-level features; missing entries masked, and set to 0.0 for the tensor library only after masking
(a placeholder, not an imputation). (3) Graded ListNet target `softmax(((2**y − 1)/(2**9 − 1)) / 0.5)`: the literal
`softmax(2**y − 1)` was measured in `NN_04` to put all mass on the top decile in float32, and temperature 1.0 was
measured in `NN_05` to be near-uniform; 0.5 was set once, never searched. (4) CPU for every measured fit (same-seed
training on MPS measured not bit-reproducible; on CPU bit-identical). (5) Early stopping on the most recent quarter
of the production training window (`training_dataset.build_training_dataset`), validation every 10 steps, patience
20, best weights restored, 1,000-step cap; five seeds (0–4).

**Measured caveats.** Under this protocol the best validation loss comes at step 10–50 on every fit in both arms;
a validation-only diagnostic shows validation loss rising monotonically after step 20 while training loss keeps
falling, and held-out ranking quality not improving with more steps — so the scored models are early-stopped
near-init models. The conditioner saw only 4–12 distinct macro states per training window. The universe is
survivorship-biased (recorded 2026-09-08), which inflates both arms equally and leaves the paired comparison
valid but no absolute number quotable. With an SE of {D.std(ddof=1) / np.sqrt(len(D)):.4f} across cycles, an effect smaller than roughly
±0.02 IC could not have been detected: the null means "not distinguishable from zero at this sample size", not
"macro conditioning is useless".

**Point-in-time.** Verified explicitly in every notebook: tensors for cycle t are byte-identical with all later
cycles removed (13 of 13 cycles; a pooled-normalization negative control fails the same audit); every training and
validation quarter precedes its target and has labels available before the target's cutoff; normalization and the
input transform use only the cycle's own cross-section; forward returns enter only the IC.

**Stop condition.** Per `BUILD_SPEC.md` §9, `NN_07` and `NN_08` were not built. Artifacts (per-fit scores/ICs, bit-
reproducible on CPU) live under `neural_ranker/outputs/` (gitignored); the executed notebooks are the record.
'''
    print(entry.strip())

## 2026-09-09 — neural ranker experiment (`neural_ranker/`): FiLM macro conditioning is null on the cached slice

**Question.** Can a macro-conditioned neural ranker (FiLM: the quarter's 6-value macro vector modulating a per-stock
encoder, `neural_ranker/OUTLINE.md` §1) express something the LambdaRank LightGBM baseline structurally cannot, and does
it show up as out-of-sample IC? Built and measured as notebooks `NN_00`–`NN_06` (+ this consolidation, `NN_09`) on the
cached real-data slice 2020-10-01 .. 2024-01-01 (14 cycles, 9 trained, 8 with a measured IC; slice fingerprint
168fedca9e7b334fdf7fb0e4b8a15cb2; git 7b7f2681146b). Nothing under `src/atlas_quant/` was modified.

**Answer: null on this slice.** Paired per seed and per cycle (same seeds, same windows, same Materials-excluded names,
same real forward returns), FiLM minus FiLM-removed IC averages -0.0094 over the 8 measured cycles
(std 0.0210, t = -1.271, p = 0.244, FiLM higher in 4 of 8 cycles; per-seed mean difference
-0.0274

## What this established

**Experiment answer, consolidated: FiLM macro conditioning is null on this slice.** Paired FiLM − no-FiLM IC over the 8 measured cycles: mean −0.0094, std 0.0210, SE 0.0074, t = −1.271, p = 0.244, FiLM higher in 4 of 8 cycles; per-seed mean difference −0.0274 .. +0.0149, std across seeds 0.0170. No production value is claimed for either arm. Every one of these numbers was read from `NN_06`'s artifact and reproduced above with the same arithmetic `NN_06` used; the stored test summary matches to 10⁻⁹ and the notebook's printed values to the shown precision.

**Walk-forward context, reproduced from `NN_05`'s artifact, not a claim.** FiLM model seed-mean IC +0.0492 (std 0.0751, hit rate 0.875) vs the stored LightGBM baseline +0.0275 (std 0.0916, hit rate 0.750); paired NN − LightGBM +0.0218 (std 0.0506), NN higher in 6 of 8; per-seed mean IC +0.0331 .. +0.0637; best early-stopping step 10 .. 50 on all 45 fits.

**Provenance verified.** Both artifacts carry this slice's fingerprint `168fedca9e7b334fdf7fb0e4b8a15cb2` and config identity, the shared fixed protocol (temperature 0.5, CPU, seeds 0–4, lr 1e-3, weight decay 1e-2, 1,000-step cap, validation every 10 steps, patience 20), 45 and 90 fit records respectively, and baseline ICs equal to the slice's own stored ICs bit for bit. All seven executed notebooks `NN_00`–`NN_06` validate with every code cell executed in order and no error output; `NN_07` and `NN_08` do not exist, by the stop condition.

**Nothing new.** This notebook trained no model, computed no new metric, and makes no claim beyond what `NN_05` and `NN_06` already printed. The draft entry above is ready to transcribe into `reproducibility_findings.md`; it is not written there by this notebook, and nothing under `src/atlas_quant/` changed anywhere in the series.